In [1]:
import os,math,warnings,math,pickle,random
warnings.filterwarnings('ignore')
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm import tqdm,tqdm_notebook

# 用于向量检索的库
import faiss

from sklearn.preprocessing import MinMaxScaler,LabelEncoder
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow.keras as keras
import copy

In [2]:
metric_recall=True # 是否要调试，如果不调试就是用全量数据集

# 1.读取数据

In [3]:
def get_all_click_sample(sample_nums=10000):
    '''debug模式，从训练集中抽取一部分数据调试代码'''
    all_click=pd.read_csv('./data/train_click_log.csv')
    all_user_ids=all_click.user_id.unique()
    sample_user_ids=np.random.choice(all_user_ids,size=sample_nums,replace=False)
    all_click=all_click[all_click['user_id'].isin(sample_user_ids)]
    all_click=all_click.drop_duplicates(['user_id','click_article_id','click_timestamp'])
    return all_click

def get_all_click_df(offline):
    '''读取点击数据，分为线上和线下，线下用训练集数据，线上用训练集+测试集'''
    if offline:
        all_click=pd.read_csv('./data/train_click_log.csv')
    else:
        trn_click=pd.read_csv('./data/train_click_log.csv')
        tst_click=pd.read_csv('./data/testA_click_log.csv')
        all_click=pd.concat([trn_click,tst_click]).reset_index(drop=True)
    all_click=all_click.drop_duplicates(['user_id','click_article_id','click_timestamp'])
    return all_click

In [4]:
def get_item_info_df():
    '''读取文章的基本属性'''
    item_info_df=pd.read_csv('./data/articles.csv')
    item_info_df=item_info_df.rename(columns={'article_id':'click_article_id'})
    return item_info_df

In [5]:
def get_item_emb_dict():
    '''读取文章的embedding'''
    item_emb_df=pd.read_csv('./data/articles_emb.csv')
    item_emb_cols=[x for x in item_emb_df.columns if 'emb' in x]
    # 把embedding转换成numpy数组，ascontiguousarray确保在内存中连续
    item_emb_np=np.ascontiguousarray(item_emb_df[item_emb_cols])
    # L2归一化
    item_emb_np=item_emb_np/np.linalg.norm(item_emb_np,axis=1,keepdims=True)
    item_emb_dict=dict(zip(item_emb_df['article_id'],item_emb_np))
    pickle.dump(item_emb_dict,open('./save/item_content_emb.pkl','wb'))
    return item_emb_dict

In [6]:
all_click_df=get_all_click_sample()
# 对时间戳归一化，按照all_click_df[['click_timestamp']]取出的是DataFrame，使用apply会对列进行操作，而all_click_df['click_timestamp']是对每个元素单独操作，取最值只会取到自己
all_click_df['click_timestamp']=all_click_df[['click_timestamp']].apply(lambda x:(x-np.min(x))/(np.max(x)-np.min(x)))

In [7]:
all_click_df.head()

,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
54,199983,162655,0.000159,4,1,17,1,16,1
55,199983,158082,0.000174,4,1,17,1,16,1
75,199980,336476,0.000904,4,3,2,1,25,2
76,199980,199198,0.000920,4,3,2,1,25,2
94,199973,156624,0.000000,4,1,17,1,16,2


In [8]:
item_info_df=get_item_info_df()

In [9]:
item_emb_dict=get_item_emb_dict()

# 2.工具函数

In [10]:
def get_user_item_time(click_df):
    '''获取用户-文章-点击时间字典 {user1:{item1:time1,item2:time2}}'''
    memo={}
    for user_id,group in click_df.groupby('user_id'):
        group_sorted=group.sort_values('click_timestamp')
        item_time_dict=dict(zip(group_sorted['click_article_id'],group_sorted['click_timestamp']))
        memo[user_id]=item_time_dict
    return memo

In [11]:
def get_item_user_time(click_df):
    '''获取文章-用户-点击时间字典  {item1: {user1: time1, user2: time2...}...}'''
    memo={}
    for click_article_id,group in click_df.groupby('click_article_id'):
        group_sorted=group.sort_values('click_timestamp')
        item_time_dict=dict(zip(group_sorted['user_id'],group_sorted['click_timestamp']))
        memo[click_article_id]=item_time_dict
    return memo

In [12]:
def get_hist_and_last_click(all_click):
    '''获取当前数据的历史点击（不包括最后一次）和最后一次点击，用于对召回结果检验'''
    all_click = all_click.sort_values(by=['user_id', 'click_timestamp'])
    click_last_df = all_click.groupby('user_id').tail(1)

    # 如果用户只有一个点击，hist为空了，会导致训练的时候这个用户不可见，此时默认泄露一下
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]

    click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)

    return click_hist_df, click_last_df

In [13]:
def get_item_info_dict(item_info_df):
    '''获取文章id对应的各个属性'''
    max_min_scaler=lambda x:(x-np.min(x))/(np.max(x)-np.min(x))
    item_info_df['created_at_ts']= item_info_df[['created_at_ts']].apply(max_min_scaler)
    item_type_dict=dict(zip(item_info_df['click_article_id'],item_info_df['category_id']))
    item_words_dict=dict(zip(item_info_df['click_article_id'],item_info_df['words_count']))
    item_created_time_dict=dict(zip(item_info_df['click_article_id'],item_info_df['created_at_ts']))
    return item_type_dict,item_words_dict,item_created_time_dict

In [14]:
def get_item_topk_click(click_df,k):
    '''获取点击次数最多的k个物品'''
    return click_df['click_article_id'].value_counts().index[:k]

In [15]:
def get_user_hist_item_info_dict(all_click):
    '''获取用户历史点击的文章信息'''

    # 获取用户user_id对应的历史点击文章类型的集合字典
    user_hist_item_types=all_click.groupby('user_id')['category_id'].agg(set).reset_index()
    user_hist_item_types_dict=dict(zip(user_hist_item_types['user_id'],user_hist_item_types['category_id']))
    # 获取user_id对应的用户点击文章的集合
    user_hist_item_ids_dict=all_click.groupby('user_id')['click_article_id'].agg(set).reset_index()
    user_hist_item_ids_dict=dict(zip(user_hist_item_ids_dict['user_id'],user_hist_item_ids_dict['click_article_id']))

    # 获取user_id对应的用户历史点击的文章的平均字数字典
    user_hist_item_words=all_click.groupby('user_id')['words_count'].agg("mean").reset_index()
    user_hist_item_words_dict=dict(zip(user_hist_item_words['user_id'],user_hist_item_words['words_count']))

    # 获取user_id对应的用户最后一次点击的文章的创建时间
    all_click_=all_click.sort_values('click_timestamp')
    user_last_item_created_time=all_click_.groupby('user_id')['created_at_ts'].apply(lambda x:x.iloc[-1]).reset_index()
    max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))
    user_last_item_created_time['created_at_ts']=user_last_item_created_time[['created_at_ts']].apply(max_min_scaler)
    user_last_item_created_time_dict=dict(zip(user_last_item_created_time['user_id'],user_last_item_created_time['created_at_ts']))

    return user_hist_item_types_dict,user_hist_item_ids_dict,user_hist_item_words_dict,user_last_item_created_time_dict

In [16]:
item_type_dict,item_words_dict,item_created_time_dict=get_item_info_dict(item_info_df)

In [17]:
# 定义一个多路召回的字典，将各路召回的结果都保存在这个字典当中
user_multi_recall_dict={'itemcf_sim_itemcf_recall':{},'embedding_sim_item_recall':{},'youtubednn_recall':{},'youtubednn_usercf_recall':{},'cold_start_recall':{}}

In [18]:
trn_hist_click_df,trn_last_click_df=get_hist_and_last_click(all_click_df)

In [19]:
def metrics_recall(user_recall_items_dict,trn_last_click_df,topk=50):
    # 对召回的结果评估
    last_click_item_dict=dict(zip(trn_last_click_df['user_id'],trn_last_click_df['click_article_id']))
    user_num=len(user_recall_items_dict)
    for k in range(10,topk+1,10):
        hit_num=0
        for user ,item_list in user_recall_items_dict.items():
            tmp=[x[0] for x in user_recall_items_dict[user][:k]]
            if last_click_item_dict[user] in set(tmp):
                hit_num+=1
        hit_rate=round(hit_num*1.0/user_num,5)
        print(' topk: ', k, ' : ', 'hit_num: ', hit_num, 'hit_rate: ', hit_rate, 'user_num : ', user_num)

# 3.计算相似度矩阵

In [20]:
def itemcf_sim(df,item_created_time_dict):
    '''计算物品的相似度矩阵,使用关联规则考虑了1. 用户点击的时间权重 2. 用户点击的顺序权重 3. 文章创建的时间权重'''
    user_item_time_dict =get_user_item_time(df)
    # i2i_sim[i][j]统计i和j共有的受众个数
    i2i_sim=defaultdict(dict)
    # 统计每个物品受众个数
    item_cnt=defaultdict(int)
    for user,item_time_list in tqdm_notebook(user_item_time_dict.items()):
        for loc1,(i,i_click_time) in enumerate(item_time_list.items()):
            # 更新
            item_cnt[i]+=1
            for loc2,(j, j_click_time) in enumerate(item_time_list.items()):
                if i!=j:
                    # 考虑文章的正向顺序点击和反向顺序点击
                    loc_alpha=1.0 if loc2>loc1 else 0.7
                    # 位置信息的权重
                    loc_weight=loc_alpha*(0.9**(np.abs(loc2-loc1)-1))
                    # 点击时间的权重
                    click_time_weight=np.exp(0.7**np.abs(i_click_time-j_click_time))
                    # 创建时间的权重
                    created_time_weight=np.exp(0.8**np.abs(item_created_time_dict[i]-item_created_time_dict[j]))
                    i2i_sim[i].setdefault(j,0)
                    # 加权弱化：用户点击的物品越多，对每对物品的贡献就越小
                    i2i_sim[i][j]+=loc_weight*click_time_weight*created_time_weight/math.log(len(item_time_list)+1)/math.log(len(item_time_list)+1)

    i2i_sim_ = defaultdict(dict)
    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j] = wij / math.sqrt(item_cnt[i] * item_cnt[j])


    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j]=wij/math.sqrt(item_cnt[i]*item_cnt[j])
    # 保存相似度矩阵
    pickle.dump(i2i_sim_, open('./save/itemcf_i2i_sim.pkl', 'wb'))
    return i2i_sim_

In [21]:
i2i_sim=itemcf_sim(all_click_df,item_created_time_dict)

  0%|          | 0/10000 [00:00<?, ?it/s]

In [22]:
all_click_df.groupby('user_id')['click_article_id'].count()

user_id
32         3
60         3
61         2
84         2
95         2
          ..
199914    15
199971     2
199973     3
199980    15
199983     2
Name: click_article_id, Length: 10000, dtype: int64

In [23]:
def get_user_activate_degree_dict(all_click_df):
    '''获取用户活跃度'''
    all_click_df_=all_click_df.groupby('user_id')['click_article_id'].count().reset_index()

    # 归一化
    mm=MinMaxScaler()
    all_click_df_['click_article_id']=mm.fit_transform(all_click_df_[['click_article_id']])

    user_activate_degree_dict = dict(zip(all_click_df_['user_id'],all_click_df_['click_article_id']))

    return user_activate_degree_dict

In [24]:
def usercf_sim(all_click_df,user_activate_degree_dict):
    '''用户相似度计算'''
    item_user_time_dict=get_item_user_time(all_click_df)
    u2u_sim=defaultdict(dict)
    user_cnt=defaultdict(int)
    for item,user_time_list in tqdm_notebook(item_user_time_dict.items()):
        for u,click_time in user_time_list.items():
            user_cnt[u]+=1
            for v,click_time in user_time_list.items():
                u2u_sim[u].setdefault(v,0)
                if u!=v:
                    activate_weight=100*0.5*(user_activate_degree_dict[u]+user_activate_degree_dict[v])
                    u2u_sim[u][v]+=activate_weight/math.log(len(user_time_list)+1)
    u2u_sim_ = copy.deepcopy(u2u_sim)

    for u,relater_users in u2u_sim.items():
        for v,wij in relater_users.items():
            u2u_sim_[u][v]=wij/math.sqrt(user_cnt[u]*user_cnt[v])
    pickle.dump(u2u_sim_,open('./save/usercf_u2u_sim.pkl', 'wb'))
    return u2u_sim_

In [25]:
user_activate_degree_dict=get_user_activate_degree_dict(all_click_df)
u2u_sim=usercf_sim(all_click_df,user_activate_degree_dict)

  0%|          | 0/6478 [00:00<?, ?it/s]

In [26]:
def embdding_sim(click_df,item_emb_df,topk):
    '''基于文章的embedding计算相似度'''

    # 文章索引与文章id的字典映射
    item_idx_2_rawid_dict=dict(zip(item_emb_df.index,item_emb_df['article_id']))

    item_emb_cols=[x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np=np.ascontiguousarray(item_emb_df[item_emb_cols].values,dtype=np.float32)
    item_emb_np=item_emb_np/np.linalg.norm(item_emb_np,axis=1,keepdims=True)

    # 建立faiss索引，基于内积
    item_index=faiss.IndexFlatIP(item_emb_np.shape[1])
    # 添加向量
    item_index.add(item_emb_np)
    # 为所有物品做一次批量检索，哈走出最相似的k个物品
    sim,idx=item_index.search(item_emb_np,topk)

    item_sim_dict=defaultdict(dict)
    for target_index,sim_value_list,rele_idx_list in tqdm_notebook(zip(range(len(item_emb_np)),sim,idx),total=len(item_emb_np)):
        # sim_value_list,rele_idx_list对当前物品所对的最相似的物品及其相似度
        target_raw_id=item_idx_2_rawid_dict[target_index]
        # 首位是物品本身
        for rele_idx,sim_value in zip(rele_idx_list[1:],sim_value_list[1:]):
            rele_raw_id=item_idx_2_rawid_dict[rele_idx]
            item_sim_dict[target_raw_id][rele_raw_id]=item_sim_dict.get(target_raw_id,{}).get(rele_raw_id,0)+sim_value

    pickle.dump(item_sim_dict,open('./save/emb_i2i_sim.pkl','wb'))
    return item_sim_dict

In [27]:
item_emb_df=pd.read_csv('./data/articles_emb.csv')
# emb_i2i_sim=embdding_sim(all_click_df,item_emb_df,topk=10)

# 4.召回

## youtubeDnn召回

In [28]:
# 获取双塔召回时的训练验证数据
# negsample指的是通过滑窗构建样本的时候，负样本的数量
def gen_data_set(data, negsample=0):
    data.sort_values("click_timestamp", inplace=True)
    item_ids = data['click_article_id'].unique()

    train_set = []
    test_set = []
    for reviewerID, hist in tqdm_notebook(data.groupby('user_id')):
        pos_list = hist['click_article_id'].tolist()

        if negsample > 0:
            candidate_set = list(set(item_ids) - set(pos_list))   # 用户没看过的文章里面选择负样本
            neg_list = np.random.choice(candidate_set,size=len(pos_list)*negsample,replace=True)  # 对于每个正样本，选择n个负样本

        # 长度只有一个的时候，需要把这条数据也放到训练集中，不然的话最终学到的embedding就会有缺失
        if len(pos_list) == 1:
            train_set.append((reviewerID, [pos_list[0]], pos_list[0],1,len(pos_list)))
            test_set.append((reviewerID, [pos_list[0]], pos_list[0],1,len(pos_list)))

        # 滑窗构造正负样本
        for i in range(1, len(pos_list)):
            hist = pos_list[:i]

            if i != len(pos_list) - 1:
                train_set.append((reviewerID, hist[::-1], pos_list[i], 1, len(hist[::-1])))  # 正样本 [user_id, his_item, pos_item, label, len(his_item)]
                for negi in range(negsample):
                    train_set.append((reviewerID, hist[::-1], neg_list[i*negsample+negi], 0,len(hist[::-1]))) # 负样本 [user_id, his_item, neg_item, label, len(his_item)]
            else:
                # 将最长的那一个序列长度作为测试数据
                test_set.append((reviewerID, hist[::-1], pos_list[i],1,len(hist[::-1])))

    random.shuffle(train_set)
    random.shuffle(test_set)

    return train_set, test_set

# 将输入的数据进行padding，使得序列特征的长度都一致
def gen_model_input(train_set,user_profile,seq_max_len):

    train_uid = np.array([line[0] for line in train_set])
    train_seq = [line[1] for line in train_set]
    train_iid = np.array([line[2] for line in train_set])
    train_label = np.array([line[3] for line in train_set])
    train_hist_len = np.array([line[4] for line in train_set])

    train_seq_pad = pad_sequences(train_seq, maxlen=seq_max_len, padding='post', truncating='post', value=0)
    train_model_input = {"user_id": train_uid, "click_article_id": train_iid, "hist_article_id": train_seq_pad,
                         "hist_len": train_hist_len}

    return train_model_input, train_label

In [29]:
from funrec.features.feature_column import FeatureColumn

print("import ok")

import ok


In [30]:
# funrec youtubeDNN召回
def youtubednn_u2i_dict(data, topk=20):
    """
    使用 FunRec 的 YouTubeDNN 两塔模型进行召回，保持与当前逻辑一致的预处理：
    - 标签/目标为正样本采样（sampled softmax 内部使用 item_id 作为 label）
    - 通过滑窗构造训练/测试样本，使用最近序列作为测试
    - 历史序列长度固定为 SEQ_LEN，并做 post-padding
    - 训练完成后提取 user/item embedding，使用 FAISS 基于内积做 TopK 近邻召回
    - 返回 {user_raw_id: [(item_raw_id, score), ...]} 的召回结果字典
    """
    import sys
    import numpy as np
    import pickle
    from tqdm import tqdm
    from sklearn.preprocessing import LabelEncoder

    from funrec.features.feature_column import FeatureColumn
    from funrec.training.trainer import train_model
    # 内联配置（参考 config_youtubednn.yaml，并适配当前数据列名）
    SEQ_LEN = 30
    emb_dim = 16
    neg_sample = 20
    dnn_units = [32]
    label_name = 'click_article_id'

    # 拷贝并做类别编码（与现有逻辑保持一致）
    df = data.copy()
    user_profile_raw = df[["user_id"]].drop_duplicates('user_id')
    item_profile_raw = df[["click_article_id"]].drop_duplicates('click_article_id')

    encoders = {}
    feature_max_idx = {}
    for col in ["user_id", "click_article_id"]:
        lbe = LabelEncoder()
        df[col] = lbe.fit_transform(df[col])
        encoders[col] = lbe
        feature_max_idx[col] = int(df[col].max()) + 1

    # 画像（仅用于 id 回退映射）
    user_profile = df[["user_id"]].drop_duplicates('user_id')
    item_profile = df[["click_article_id"]].drop_duplicates('click_article_id')
    user_index_2_rawid = dict(zip(user_profile['user_id'], user_profile_raw['user_id']))
    item_index_2_rawid = dict(zip(item_profile['click_article_id'], item_profile_raw['click_article_id']))

    # 按当前逻辑构造训练/测试样本
    train_set, test_set = gen_data_set(df, 0)
    train_model_input, _ = gen_model_input(train_set, user_profile, SEQ_LEN)
    test_model_input, _ = gen_model_input(test_set, user_profile, SEQ_LEN)

    # 仅保留模型实际需要的输入键
    input_keys = ['user_id', 'click_article_id', 'hist_article_id']
    train_X = {k: np.asarray(train_model_input[k], dtype=np.int32) for k in input_keys}
    test_X = {k: np.asarray(test_model_input[k], dtype=np.int32) for k in input_keys}

    # 手动定义特征列（不依赖外部数据字典）
    feature_columns = [
        FeatureColumn(name='user_id', group=['user_dnn'], type='sparse', vocab_size=feature_max_idx['user_id'], emb_dim=emb_dim),
        FeatureColumn(name='click_article_id', group=['target_item'], type='sparse', vocab_size=feature_max_idx['click_article_id'], emb_dim=emb_dim),
        FeatureColumn(name='hist_article_id', emb_name='click_article_id', group=['raw_hist_seq'], type='varlen_sparse', max_len=SEQ_LEN, combiner='mean', emb_dim=emb_dim, vocab_size=feature_max_idx['click_article_id']),
    ]

    # 组装 processed_data（与 FunRec 训练器期望的结构一致）
    processed_data = {
        'train': {
            'features': train_X,
            'labels': None  # 由 positive_sampling_labels 规则内部替换为全 1
        },
        'test': {
            'features': test_X,
            'labels': None,
            'eval_data': {}
        },
        'all_items': {
            'click_article_id': np.arange(feature_max_idx['click_article_id'], dtype=np.int32)
        },
        'feature_dict': {
            'user_id': feature_max_idx['user_id'],
            'click_article_id': feature_max_idx['click_article_id']
        }
    }

    # 训练配置（内联）
    training_config = {
        'build_function': 'funrec.models.youtubednn.build_youtubednn_model',
        'data_preprocessing': [
            {'type': 'positive_sampling_labels'}
        ],
        'model_params': {
            'emb_dim': emb_dim,
            'neg_sample': neg_sample,
            'dnn_units': dnn_units,
            'label_name': label_name
        },
        'optimizer': 'adam',
        'optimizer_params': {
            'learning_rate': 1e-4
        },
        'loss': 'sampledsoftmaxloss',
        'batch_size': 128,
        'epochs': 5,
        'verbose': 0
    }

    # 训练模型（返回 main_model, user_model, item_model）
    model, user_model, item_model = train_model(training_config, feature_columns, processed_data)

    # 提取 embedding
    user_inputs_for_pred = {k: test_X[k] for k in ['user_id', 'hist_article_id']}
    user_embs = user_model.predict(user_inputs_for_pred, batch_size=2 ** 12, verbose=0)
    item_embs = item_model.predict(processed_data['all_items'], batch_size=2 ** 12, verbose=0)

    # 归一化（与现有逻辑一致）
    user_embs = user_embs / np.linalg.norm(user_embs, axis=1, keepdims=True)
    item_embs = item_embs / np.linalg.norm(item_embs, axis=1, keepdims=True)

    # 保存 embedding（与现有逻辑一致，注意 id 回退）
    raw_user_id_emb_dict = {user_index_2_rawid[k]: v for k, v in zip(test_X['user_id'], user_embs)}
    raw_item_id_emb_dict = {item_index_2_rawid[k]: v for k, v in zip(processed_data['all_items']['click_article_id'], item_embs)}
    pickle.dump(raw_user_id_emb_dict, open('./save/user_youtube_emb.pkl', 'wb'))
    pickle.dump(raw_item_id_emb_dict, open('./save/item_youtube_emb.pkl', 'wb'))

    # 使用 FAISS 做向量检索召回
    index = faiss.IndexFlatIP(emb_dim)
    index.add(item_embs.astype(np.float32))
    sim, idx = index.search(np.ascontiguousarray(user_embs.astype(np.float32)), topk)

    user_recall_items_dict = defaultdict(dict)
    for target_idx, sim_value_list, rele_idx_list in tqdm_notebook(zip(test_X['user_id'], sim, idx)):
        target_raw_id = user_index_2_rawid[int(target_idx)]
        # 从 1 开始去掉最相似的第一个（通常为本身或极近邻）
        for rele_idx, sim_value in zip(rele_idx_list[1:], sim_value_list[1:]):
            rele_raw_id = item_index_2_rawid[int(rele_idx)]
            user_recall_items_dict[target_raw_id][rele_raw_id] = user_recall_items_dict.get(target_raw_id, {}).get(rele_raw_id, 0) + float(sim_value)

    # 排序并保存
    user_recall_items_dict = {k: sorted(v.items(), key=lambda x: x[1], reverse=True) for k, v in user_recall_items_dict.items()}
    pickle.dump(user_recall_items_dict, open('./save/youtube_u2i_dict.pkl', 'wb'))
    return user_recall_items_dict

In [31]:
# 由于这里需要做召回评估，所以讲训练集中的最后一次点击都提取了出来
if not metric_recall:
    user_multi_recall_dict['youtubednn_recall'] = youtubednn_u2i_dict(all_click_df, topk=20)
else:
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
    user_multi_recall_dict['youtubednn_recall'] = youtubednn_u2i_dict(trn_hist_click_df, topk=20)
    # 召回效果评估
    metrics_recall(user_multi_recall_dict['youtubednn_recall'], trn_last_click_df, topk=20)

  0%|          | 0/10000 [00:00<?, ?it/s]

0it [00:00, ?it/s]

 topk:  10  :  hit_num:  521 hit_rate:  0.0521 user_num :  10000
 topk:  20  :  hit_num:  942 hit_rate:  0.0942 user_num :  10000


## itemCF recall
召回中使用了关联规则
+ 考虑相似文章与历史点击文章顺序的权重
+ 考虑文章创建时间的权重，也就是考虑相似文章与历史点击文章创建时间差的权重
+ 考虑文章内容相似度权重(使用Embedding计算相似文章相似度，但是这里需要注意，在Embedding的时候并没有计算所有商品两两之间的相似度，所以相似的文章与历史点击文章不存在相似度，需要做特殊处理)


In [32]:
def item_based_recommend(user_id,user_item_time_dict,i2i_sim,sim_item_topk,recall_item_num,item_topk_click,item_created_time_dict,emb_i2i_sim):
    """
    基于文章的协同过滤召回
    :param user_id: 用户id
    :param user_item_time_dict: 字典  {user1: {item1: time1, item2: time2..}...}按照时间排序
    :param i2i_sim: 物品相似度矩阵
    :param sim_item_topk: 选择与物品最相似的钱k篇文章
    :param recall_item_num: 最后召回的数量
    :param item_topk_click: 点击次数最多的文章列表，永不补全召回
    :param item_created_time_dict:
    :param emb_i2i_sim: 基于embedding的物品相似度矩阵
    :return:召回的文章列表 {item1:score1, item2: score2...}
    """
    # 获取用户历史交互的文章
    user_hist_items=user_item_time_dict[user_id]

    item_rank={}
    for loc ,(i,click_time) in enumerate(user_hist_items.items()):
        for j,wij in sorted(i2i_sim[i].items(),key=lambda x:x[1],reverse=True)[:sim_item_topk]:
            if j not in user_hist_items:
                # 文章创建时间差权重，时间差越大权重越小，用户更倾向于发布时间按相近的文章
                created_time_weight=np.exp(0.8**np.abs(item_created_time_dict[i]-item_created_time_dict[j]))
                # 位置权重 靠后夫人文章更能代表用户最近的兴趣
                loc_weight=(0.9**(len(user_hist_items)-loc))

                content_weight=1.0
                # 两篇文章越相似权重越大
                content_weight += emb_i2i_sim.get(i, {}).get(j, 0)
                content_weight += emb_i2i_sim.get(j, {}).get(i, 0)

                item_rank.setdefault(j,0)
                item_rank[j]+=created_time_weight*loc_weight*content_weight*wij
    if len(item_rank)<recall_item_num:
        for i,item in enumerate(item_topk_click):
            if item not in item_rank:# 要填充的不在原来的列表中
                item_rank[item]=-i-100 # 随机负数
                if len(item_rank)==recall_item_num:break
    item_rank=sorted(item_rank.items(),key=lambda x:x[1],reverse=True)[:recall_item_num]
    return item_rank

itemCF sim召回

In [33]:
# 判断是否需要检验
if metric_recall:
    trn_hist_click_df,trn_last_click_df=get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df=all_click_df

# 存储结果的字典
user_recall_items_dict=defaultdict(dict)
#  {user1: {item1: time1, item2: time2..}...}
user_item_time_dict=get_user_item_time(trn_hist_click_df)

i2i_sim=pickle.load(open('./save/itemcf_i2i_sim.pkl','rb'))
emb_i2i_sim=pickle.load(open('./save/emb_i2i_sim.pkl','rb'))

sim_item_topk=20
recall_item_num=10
# 点击次数最多的物品
item_topk_click=get_item_topk_click(trn_hist_click_df,k=50)

for user in tqdm_notebook(trn_hist_click_df['user_id'].unique(),):
    user_recall_items_dict[user]=item_based_recommend(user,user_item_time_dict, i2i_sim, sim_item_topk, recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)

# 存储itemCF sim召回的结果
user_multi_recall_dict['itemcf_sim_itemcf_recall']=user_recall_items_dict
pickle.dump(user_multi_recall_dict['itemcf_sim_itemcf_recall'],open('./save/itemcf_recall_dict.pkl','wb'))

if metric_recall:
    metrics_recall(user_multi_recall_dict['itemcf_sim_itemcf_recall'],trn_last_click_df,topk=recall_item_num)

  0%|          | 0/10000 [00:00<?, ?it/s]

 topk:  10  :  hit_num:  6512 hit_rate:  0.6512 user_num :  10000


embedding sim 召回

In [56]:
if metric_recall:
    trn_hist_click_df,trn_last_click_df=get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df=all_click_df
user_recall_items_dict=defaultdict(dict)
user_item_time_dict=get_user_item_time(trn_hist_click_df)

# 改一下相似度矩阵即可
i2i_sim=pickle.load(open('./save/emb_i2i_sim.pkl','rb'))

sim_item_topk=20
recall_item_num=10
item_topk_click=get_item_topk_click(trn_hist_click_df,k=50)

for user in tqdm_notebook(trn_hist_click_df['user_id'].unique(),):
    user_recall_items_dict[user]=item_based_recommend(user,user_item_time_dict, i2i_sim, sim_item_topk, recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)

user_multi_recall_dict['embedding_sim_item_recall']=user_recall_items_dict
pickle.dump(user_multi_recall_dict['embedding_sim_item_recall'],open('./save/embedding_sim_item_recall.pkl','wb'))

if metric_recall:
    metrics_recall(user_multi_recall_dict['embedding_sim_item_recall'],trn_last_click_df,topk=recall_item_num)

  0%|          | 0/10000 [00:00<?, ?it/s]

 topk:  10  :  hit_num:  205 hit_rate:  0.0205 user_num :  10000


userCF召回

In [35]:
def user_based_recommand(user_id,user_item_time_dict,u2u_sim,sim_user_topk,recall_item_num,item_topk_click,item_created_time_dict,emb_i2i_sim):
    """
    基于用户的召回u2u2i
    :param user_id: 用户id
    :param user_item_time_dict:  {user1: {item1: time1, item2: time2..}...}
    :param u2u_sim: 用户相似度矩阵
    :param sim_user_topk: 选择与当前用户最相似的前k个用户
    :param recall_item_num: 最后的召回文章数量
    :param item_topk_click: 点击次数最多的文章列表，用户召回补全
    :param item_created_time_dict: 文章创建时间列表
    :param emb_i2i_sim:内容embedding的相似矩阵
    :return: 召回的文章列表 {item1:score1, item2: score2...}
    """

    # 历史交互
    user_item_time_list=user_item_time_dict[user_id]
    user_hist_items=set([i for i,_ in user_item_time_list.items()])

    items_rank={}
    # 相似用户
    for sim_u,wuv in sorted(u2u_sim[user_id].items(),key=lambda x:x[1],reverse=True)[:sim_user_topk]:
        # 相似用户交互过的物品
        for i,click_time in user_item_time_dict[sim_u].items():
            if i not in user_hist_items:
                items_rank.setdefault(i,0)

                loc_weight=1.0
                content_weight=1.0
                created_time_weight=1.0
                # 该物品与当前用户的历史物品做权重交互
                for loc,(j,click_time) in enumerate(user_item_time_list.items()):
                    loc_weight+=0.9**(len(user_item_time_list)-loc)
                    content_weight+=emb_i2i_sim.get(i,{}).get(j,0)
                    content_weight+=emb_i2i_sim.get(j,{}).get(i,0)
                    created_time_weight+=np.exp(0.8*np.abs(item_created_time_dict[i]-item_created_time_dict[j]))

                items_rank[i]+=loc_weight*content_weight*created_time_weight*wuv
    if len(items_rank)<recall_item_num:
        for i,item in enumerate(item_topk_click):
            if item not in items_rank:
                items_rank[item]=-i-100
                if len(items_rank)==recall_item_num:break
    items_rank=sorted(items_rank.items(),key=lambda x:x[1],reverse=True)[:recall_item_num]

    return items_rank


In [36]:
if metric_recall:
    trn_hist_click_df,trn_last_click_df=get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df=all_click_df
user_recall_items_dict=defaultdict(dict)
user_item_time_dict=get_user_item_time(trn_hist_click_df)

# 改一下相似度矩阵即可
u2u_sim=pickle.load(open('./save/usercf_u2u_sim.pkl','rb'))

sim_item_topk=20
recall_item_num=10
item_topk_click=get_item_topk_click(trn_hist_click_df,k=50)

for user in tqdm_notebook(trn_hist_click_df['user_id'].unique(),):
    user_recall_items_dict[user]=user_based_recommand(user,user_item_time_dict, u2u_sim, sim_item_topk, recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)

pickle.dump(user_recall_items_dict,open('./save/usercf_u2u2i_recall.pkl','wb'))

if metric_recall:
    metrics_recall(user_recall_items_dict,trn_last_click_df,topk=recall_item_num)

  0%|          | 0/10000 [00:00<?, ?it/s]

 topk:  10  :  hit_num:  6744 hit_rate:  0.6744 user_num :  10000


user embedding sim召回，使用youtubeDNN得到的用户embedding进行向量检索

In [60]:
def u2u_embedding_sim(user_emb_dict,topk):
    """使用embedding的方式获取u2u的相似度矩阵"""
    user_list=[]  # 用户id列表
    user_emb_list=[] # 用户embedding列表
    for user_id,user_emb in user_emb_dict.items():
        user_list.append(user_id)
        user_emb_list.append(user_emb)

    # 索引与用户id映射
    user_idx_2_rawid_dict=dict([(i,x) for i,x in enumerate(user_list)])
    user_emb_np=np.array(user_emb_list,dtype=np.float32)

    # 建立向量检索
    user_index=faiss.IndexFlatL2(user_emb_np.shape[1])
    user_index.add(user_emb_np)
    sim,idx=user_index.search(user_emb_np,topk)

    user_sim_dict=defaultdict(dict)
    # target_idx 当前用户,sim_value_list与该用户向量邻近的相似度,rele_idx_list 对应的相似用户索引
    for target_idx,sim_value_list,rele_idx_list in tqdm_notebook(zip(range(len(user_emb_np)),sim,idx)):
        target_raw_id=user_idx_2_rawid_dict[target_idx]
        for rele_idx,sim_value in zip(rele_idx_list[1:],sim_value_list[1:]):
            rele_raw_id=user_idx_2_rawid_dict[rele_idx]
            user_sim_dict[target_raw_id][rele_raw_id]=user_sim_dict.get(target_raw_id,{}).get(rele_raw_id,0)+sim_value

    # 保存
    pickle.dump(user_sim_dict,open('./save/youtube_u2u_sim.pkl','wb'))
    return user_sim_dict

In [61]:
# 这里的user embedding的表现不是很好，因为youtubeDNN使用用户点击序列来训练embedding,但是这里的点击学历都比较短
user_emb_dict=pickle.load(open('./save/user_youtube_emb.pkl','rb'))
u2u_sim=u2u_embedding_sim(user_emb_dict,topk=10)

0it [00:00, ?it/s]

In [63]:
if metric_recall:
    trn_hist_click_df,trn_last_click_df=get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df=all_click_df
user_recall_items_dict=defaultdict(dict)
user_item_time_dict=get_user_item_time(trn_hist_click_df)

# 改一下相似度矩阵即可
u2u_sim=pickle.load(open('./save/youtube_u2u_sim.pkl','rb'))

sim_user_topk=20
recall_item_num=10
item_topk_click=get_item_topk_click(trn_hist_click_df,k=50)

for user in tqdm_notebook(trn_hist_click_df['user_id'].unique()):
    user_recall_items_dict[user]=user_based_recommand(user,user_item_time_dict, u2u_sim, sim_user_topk, recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)

pickle.dump(user_recall_items_dict,open('./save/usercf_u2u2i_recall.pkl','wb'))

if metric_recall:
    metrics_recall(user_recall_items_dict,trn_last_click_df,topk=recall_item_num)

  0%|          | 0/10000 [00:00<?, ?it/s]

 topk:  10  :  hit_num:  284 hit_rate:  0.0284 user_num :  10000


# 5.冷启动
本题场景下，数据集中有30W的文章，但是点击数据只有3W，因此文章存在冷启动问题，同理用户有出现只点击过一次的记录，因此用户也需要冷启动，这里只解决物品冷启动。